In [2]:
"""
threshold_sensitivity.py
========================
Runs full validation at each diffuse fraction threshold
using pre-processed SURFRAD folders and r.sun station values.

Reads:
  surfrad_processed_0.20/surfrad_multiyear_means.csv
  surfrad_processed_0.25/surfrad_multiyear_means.csv
  surfrad_processed_0.30/surfrad_multiyear_means.csv
  surfrad_processed_0.35/surfrad_multiyear_means.csv
  surfrad_processed_0.40/surfrad_multiyear_means.csv

Outputs per threshold:
  validation_outputs/threshold_0.XX/validation_stats.csv
  validation_outputs/threshold_0.XX/station_bias.csv
  validation_outputs/threshold_0.XX/Fig1_scatter.png

Summary outputs:
  validation_outputs/threshold_sensitivity_stats.csv
  validation_outputs/Fig5_threshold_sensitivity.png
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats as scipy_stats
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================
BASE       = r"C:\Users\mgvhy\OneDrive - University of Missouri\scientific_data\analysis"
RSUN_FILE  = os.path.join(BASE, "rsun_station_values.csv")
OUT_BASE   = os.path.join(BASE, "validation_outputs")

THRESHOLDS = [0.20, 0.25, 0.30, 0.35, 0.40]
COMPONENTS  = ["glob_rad", "beam_rad", "diff_rad"]
COMP_LABELS = {
    "glob_rad": "Global (glob_rad)",
    "beam_rad": "Beam (beam_rad)",
    "diff_rad": "Diffuse (diff_rad)",
}
COMP_COLORS = {
    "glob_rad": "#E41A1C",
    "beam_rad": "#377EB8",
    "diff_rad": "#4DAF4A",
}
STATION_COLORS = {
    "bon":"#1b9e77","fpk":"#d95f02","gwn":"#7570b3",
    "tbl":"#e7298a","dra":"#66a61e","psu":"#e6ab02","sxf":"#a6761d",
}
STATION_MARKERS = {
    "bon":"o","fpk":"^","gwn":"s","tbl":"D","dra":"*","psu":"+","sxf":"x",
}

# ============================================================
# HELPERS
# ============================================================

def load_surfrad(means_path, daily_path):
    surfrad = pd.read_csv(means_path)
    surfrad["doy"] = surfrad["doy"].astype(int)
    surf_long = surfrad.melt(
        id_vars=["station","doy"],
        value_vars=["glob_rad_mean","beam_rad_mean","diff_rad_mean"],
        var_name="component", value_name="surfrad_Whm2"
    )
    surf_long["component"] = surf_long["component"].str.replace("_mean","",regex=False)
    surf_std = surfrad.melt(
        id_vars=["station","doy"],
        value_vars=["glob_rad_std","beam_rad_std","diff_rad_std"],
        var_name="component", value_name="surfrad_sd"
    )
    surf_std["component"] = surf_std["component"].str.replace("_std","",regex=False)
    surf_long = surf_long.merge(surf_std, on=["station","doy","component"], how="left")
    n_daily = len(pd.read_csv(daily_path))
    return surf_long, n_daily


def compute_stats(val):
    results = []
    for comp in COMPONENTS:
        sub = val[val["component"] == comp].dropna(subset=["rsun_Whm2","surfrad_Whm2"])
        if len(sub) < 5:
            continue
        bias     = sub["rsun_Whm2"] - sub["surfrad_Whm2"]
        rmse     = np.sqrt(np.mean(bias**2))
        mbe      = np.mean(bias)
        r2       = np.corrcoef(sub["rsun_Whm2"], sub["surfrad_Whm2"])[0,1]**2
        mean_obs = np.mean(sub["surfrad_Whm2"])
        sl, ic, *_ = scipy_stats.linregress(sub["surfrad_Whm2"], sub["rsun_Whm2"])
        results.append({
            "component"  : comp,
            "n"          : len(sub),
            "R2"         : round(r2,   4),
            "RMSE"       : round(rmse, 1),
            "MBE"        : round(mbe,  1),
            "rRMSE_pct"  : round(rmse/mean_obs*100, 1),
            "rMBE_pct"   : round(mbe /mean_obs*100, 1),
            "slope"      : round(sl,   4),
            "intercept"  : round(ic,   1),
        })
    return pd.DataFrame(results)


def make_scatter(val, stats_df, thresh, out_dir):
    ax_max = max(val["rsun_Whm2"].max(), val["surfrad_Whm2"].max()) * 1.05
    ax_min = 0
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
    for ax, comp in zip(axes, COMPONENTS):
        sub = val[val["component"] == comp].copy()
        row = stats_df[stats_df["component"] == comp].iloc[0]
        ax.plot([ax_min,ax_max],[ax_min,ax_max],
                color="gray",linestyle="--",linewidth=1,zorder=1)
        x_line = np.array([ax_min, ax_max])
        ax.plot(x_line, row["slope"]*x_line+row["intercept"],
                color="black",linewidth=1,zorder=2)
        for st in sorted(sub["station"].unique()):
            s = sub[sub["station"]==st]
            ax.scatter(s["surfrad_Whm2"], s["rsun_Whm2"],
                       color=STATION_COLORS.get(st,"gray"),
                       marker=STATION_MARKERS.get(st,"o"),
                       alpha=0.85, s=55, label=st.upper(), zorder=3)
        ann = (f"R²={row['R2']:.3f}\n"
               f"RMSE={row['RMSE']:.0f} Wh/m²\n"
               f"MBE={row['MBE']:.0f} Wh/m²")
        ax.text(0.05,0.97,ann,transform=ax.transAxes,
                fontsize=9,va="top",ha="left",
                bbox=dict(boxstyle="round,pad=0.3",facecolor="white",alpha=0.85))
        ax.set_xlim(ax_min, ax_max)
        ax.set_ylim(ax_min, ax_max)
        ax.set_xlabel("SURFRAD measured (Wh m⁻² day⁻¹)")
        ax.set_ylabel("r.sun modeled (Wh m⁻² day⁻¹)")
        ax.set_title(COMP_LABELS[comp])
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
        ax.set_aspect("equal")
        ax.grid(True, alpha=0.3)
    handles, labels = axes[-1].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=7,
               fontsize=9, title="Station", bbox_to_anchor=(0.5,-0.08))
    fig.suptitle(f"r.sun vs SURFRAD — DHI/GHI threshold = {thresh:.2f}",
                 fontsize=13, fontweight="bold")
    plt.savefig(os.path.join(out_dir, "Fig1_scatter.png"),
                dpi=300, bbox_inches="tight")
    plt.close()


# ============================================================
# MAIN LOOP
# ============================================================
print("Loading r.sun station values...")
rsun = pd.read_csv(RSUN_FILE)
rsun = rsun[rsun["doy"] != "annual"].copy()
rsun["doy"]       = rsun["doy"].astype(int)
rsun["rsun_Whm2"] = pd.to_numeric(rsun["rsun_Whm2"], errors="coerce")
rsun = rsun[["station","doy","component","rsun_Whm2"]].copy()

all_summary = []

for thresh in THRESHOLDS:
    folder     = f"surfrad_processed_{thresh:.2f}"
    means_path = os.path.join(BASE, folder, "surfrad_multiyear_means.csv")
    daily_path = os.path.join(BASE, folder, "surfrad_daily_clearsky.csv")
    out_dir    = os.path.join(OUT_BASE, f"threshold_{thresh:.2f}")
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    if not os.path.exists(means_path) or not os.path.exists(daily_path):
        print(f"MISSING: {folder} — skipping")
        continue

    print(f"\n{'='*55}")
    print(f"  Threshold = {thresh:.2f}")
    print(f"{'='*55}")

    surf_long, n_daily = load_surfrad(means_path, daily_path)

    # Join
    val = rsun.merge(surf_long, on=["station","doy","component"], how="inner")
    val = val.dropna(subset=["rsun_Whm2","surfrad_Whm2"])
    val["bias"] = val["rsun_Whm2"] - val["surfrad_Whm2"]

    # Stats
    stats_df = compute_stats(val)
    stats_df["threshold"] = thresh
    stats_df["n_daily"]   = n_daily
    stats_df.to_csv(os.path.join(out_dir,"validation_stats.csv"), index=False)

    # Per-station bias
    station_bias = val.groupby(["station","component"],group_keys=False).apply(
        lambda g: pd.Series({
            "n"        : len(g),
            "MBE"      : round(np.mean(g["bias"]),1),
            "RMSE"     : round(np.sqrt(np.mean(g["bias"]**2)),1),
            "R2"       : round(np.corrcoef(g["rsun_Whm2"],g["surfrad_Whm2"])[0,1]**2,4),
            "rMBE_pct" : round(np.mean(g["bias"])/np.mean(g["surfrad_Whm2"])*100,1),
        })
    ).reset_index()
    station_bias.to_csv(os.path.join(out_dir,"station_bias.csv"), index=False)

    # Scatter plot
    make_scatter(val, stats_df, thresh, out_dir)
    print(f"  n_daily={n_daily}  n_pairs={len(val)//3}")

    # Print stats
    print(f"  {'Component':>22} {'R2':>7} {'RMSE':>8} {'MBE':>8} {'rMBE%':>8}")
    for _, r in stats_df.iterrows():
        print(f"  {COMP_LABELS[r['component']]:>22} {r['R2']:>7.3f} "
              f"{r['RMSE']:>8.0f} {r['MBE']:>8.0f} {r['rMBE_pct']:>8.1f}%")

    for _, r in stats_df.iterrows():
        all_summary.append({**r.to_dict(), "threshold": thresh, "n_daily": n_daily})

# ============================================================
# SUMMARY TABLE
# ============================================================
df_all = pd.DataFrame(all_summary)
df_all.to_csv(os.path.join(OUT_BASE,"threshold_sensitivity_stats.csv"), index=False)
print(f"\nSaved: threshold_sensitivity_stats.csv")

# ============================================================
# FIG 5 — 4-panel sensitivity figure
# ============================================================
print("Generating Fig5_threshold_sensitivity.png...")
fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)

# R²
ax = axes[0,0]
for comp in COMPONENTS:
    sub = df_all[df_all["component"]==comp].sort_values("threshold")
    ax.plot(sub["threshold"], sub["R2"], color=COMP_COLORS[comp],
            linewidth=2, marker="o", markersize=8, label=COMP_LABELS[comp])
ax.axvline(0.30,color="gray",linestyle="--",linewidth=1.5,label="0.30 (selected)")
ax.set_xlabel("DHI/GHI threshold"); ax.set_ylabel("R²")
ax.set_title("R² by component", fontweight="bold")
ax.set_xticks(THRESHOLDS); ax.legend(fontsize=9); ax.grid(True,alpha=0.3)

# rMBE
ax = axes[0,1]
for comp in COMPONENTS:
    sub = df_all[df_all["component"]==comp].sort_values("threshold")
    ax.plot(sub["threshold"], sub["rMBE_pct"], color=COMP_COLORS[comp],
            linewidth=2, marker="o", markersize=8, label=COMP_LABELS[comp])
ax.axvline(0.30,color="gray",linestyle="--",linewidth=1.5,label="0.30 (selected)")
ax.set_xlabel("DHI/GHI threshold"); ax.set_ylabel("rMBE (%)")
ax.set_title("Relative MBE (%) by component", fontweight="bold")
ax.set_xticks(THRESHOLDS); ax.legend(fontsize=9); ax.grid(True,alpha=0.3)

# RMSE
ax = axes[1,0]
for comp in COMPONENTS:
    sub = df_all[df_all["component"]==comp].sort_values("threshold")
    ax.plot(sub["threshold"], sub["RMSE"], color=COMP_COLORS[comp],
            linewidth=2, marker="o", markersize=8, label=COMP_LABELS[comp])
ax.axvline(0.30,color="gray",linestyle="--",linewidth=1.5,label="0.30 (selected)")
ax.set_xlabel("DHI/GHI threshold"); ax.set_ylabel("RMSE (Wh m⁻² day⁻¹)")
ax.set_title("RMSE by component", fontweight="bold")
ax.set_xticks(THRESHOLDS)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
ax.legend(fontsize=9); ax.grid(True,alpha=0.3)

# Sample size
ax = axes[1,1]
sub_n = df_all[df_all["component"]=="glob_rad"].sort_values("threshold")
bars = ax.bar(sub_n["threshold"], sub_n["n_daily"],
              width=0.04, color="steelblue", alpha=0.85, edgecolor="white")
ax.axvline(0.30,color="gray",linestyle="--",linewidth=1.5,label="0.30 (selected)")
for bar, val in zip(bars, sub_n["n_daily"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+3,
            str(int(val)), ha="center", va="bottom", fontsize=9)
ax.set_xlabel("DHI/GHI threshold"); ax.set_ylabel("Valid station-year-DOY records")
ax.set_title("Sample size by threshold", fontweight="bold")
ax.set_xticks(THRESHOLDS); ax.legend(fontsize=9); ax.grid(True,axis="y",alpha=0.3)

fig.suptitle(
    "Sensitivity of validation statistics to diffuse fraction threshold\n"
    "SURFRAD 7 stations × 12 DOYs × 2015–2024",
    fontsize=12, fontweight="bold"
)
plt.savefig(os.path.join(OUT_BASE,"Fig5_threshold_sensitivity.png"),
            dpi=300, bbox_inches="tight")
plt.close()
print("Saved: Fig5_threshold_sensitivity.png")

# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n=== FINAL SUMMARY — glob_rad ===")
glob = df_all[df_all["component"]=="glob_rad"].sort_values("threshold")
print(glob[["threshold","n_daily","R2","rMBE_pct","RMSE"]].to_string(index=False))
print(f"\nMonotonic R² increase: {glob['R2'].is_monotonic_increasing}")
print(f"Monotonic rMBE decrease: {glob['rMBE_pct'].is_monotonic_decreasing}")

Loading r.sun station values...

  Threshold = 0.20
  n_daily=472  n_pairs=83
               Component      R2     RMSE      MBE    rMBE%
       Global (glob_rad)   0.483     3239     2752     86.5%
         Beam (beam_rad)   0.475     2714     2227     81.0%
      Diffuse (diff_rad)   0.516      548      523    121.8%

  Threshold = 0.25
  n_daily=546  n_pairs=84
               Component      R2     RMSE      MBE    rMBE%
       Global (glob_rad)   0.531     3015     2530     75.3%
         Beam (beam_rad)   0.512     2572     2085     73.0%
      Diffuse (diff_rad)   0.609      468      444     87.9%

  Threshold = 0.30
  n_daily=590  n_pairs=84
               Component      R2     RMSE      MBE    rMBE%
       Global (glob_rad)   0.596     2858     2407     69.1%
         Beam (beam_rad)   0.563     2485     2025     69.4%
      Diffuse (diff_rad)   0.696      403      380     67.0%

  Threshold = 0.35
  n_daily=618  n_pairs=84
               Component      R2     RMSE      MBE    r

In [4]:
import pandas as pd
import numpy as np
from scipy import stats

BASE = r"C:\Users\mgvhy\OneDrive - University of Missouri\scientific_data\analysis"

# Load r.sun
rsun = pd.read_csv(f"{BASE}/rsun_station_values.csv")
rsun = rsun[rsun["doy"] != "annual"].copy()
rsun["doy"] = rsun["doy"].astype(int)
rsun["rsun_Whm2"] = pd.to_numeric(rsun["rsun_Whm2"], errors="coerce")
rsun = rsun[["station","doy","component","rsun_Whm2"]].copy()

# Load SURFRAD means — add lat to id_vars
surfrad = pd.read_csv(f"{BASE}/surfrad_processed_0.30/surfrad_multiyear_means.csv")
surfrad["doy"] = surfrad["doy"].astype(int)

surf_long = surfrad.melt(
    id_vars=["station","doy","lat"],          # ← lat included here
    value_vars=["glob_rad_mean"],
    var_name="component", value_name="surfrad_Whm2"
)
surf_long["component"] = "glob_rad"

# Join
val = rsun[rsun["component"]=="glob_rad"].merge(
    surf_long, on=["station","doy","component"], how="inner"
).dropna()

print(f"Joined rows: {len(val)}")
print(f"Columns: {list(val.columns)}")

# Decompose variance
r2_doy_rsun    = stats.pearsonr(val["doy"], val["rsun_Whm2"])[0]**2
r2_doy_surfrad = stats.pearsonr(val["doy"], val["surfrad_Whm2"])[0]**2
r2_lat_rsun    = stats.pearsonr(val["lat"], val["rsun_Whm2"])[0]**2
r2_lat_surfrad = stats.pearsonr(val["lat"], val["surfrad_Whm2"])[0]**2
r2_overall     = np.corrcoef(val["rsun_Whm2"], val["surfrad_Whm2"])[0,1]**2

val["rsun_norm"]    = (val["rsun_Whm2"]    - val["rsun_Whm2"].mean())    / val["rsun_Whm2"].std()
val["surfrad_norm"] = (val["surfrad_Whm2"] - val["surfrad_Whm2"].mean()) / val["surfrad_Whm2"].std()
r2_norm = np.corrcoef(val["rsun_norm"], val["surfrad_norm"])[0,1]**2

print("\n=== VARIANCE DECOMPOSITION ===")
print(f"R²(DOY  → r.sun)    = {r2_doy_rsun:.3f}   seasonal signal in model")
print(f"R²(DOY  → SURFRAD)  = {r2_doy_surfrad:.3f}   seasonal signal in observations")
print(f"R²(lat  → r.sun)    = {r2_lat_rsun:.3f}   spatial signal in model")
print(f"R²(lat  → SURFRAD)  = {r2_lat_surfrad:.3f}   spatial signal in observations")
print(f"R²(model vs obs)    = {r2_overall:.3f}   overall validation R²")
print(f"R²(normalized)      = {r2_norm:.3f}   pattern correlation (bias removed)")

print("\n=== SEASONAL RANGE ===")
print(f"r.sun   seasonal range: {val.groupby('doy')['rsun_Whm2'].mean().max() - val.groupby('doy')['rsun_Whm2'].mean().min():.0f} Wh/m²")
print(f"SURFRAD seasonal range: {val.groupby('doy')['surfrad_Whm2'].mean().max() - val.groupby('doy')['surfrad_Whm2'].mean().min():.0f} Wh/m²")

print("\n=== SPATIAL RANGE ===")
print(f"r.sun   spatial range: {val.groupby('station')['rsun_Whm2'].mean().max() - val.groupby('station')['rsun_Whm2'].mean().min():.0f} Wh/m²")
print(f"SURFRAD spatial range: {val.groupby('station')['surfrad_Whm2'].mean().max() - val.groupby('station')['surfrad_Whm2'].mean().min():.0f} Wh/m²")

Joined rows: 84
Columns: ['station', 'doy', 'component', 'rsun_Whm2', 'lat', 'surfrad_Whm2']

=== VARIANCE DECOMPOSITION ===
R²(DOY  → r.sun)    = 0.018   seasonal signal in model
R²(DOY  → SURFRAD)  = 0.009   seasonal signal in observations
R²(lat  → r.sun)    = 0.029   spatial signal in model
R²(lat  → SURFRAD)  = 0.080   spatial signal in observations
R²(model vs obs)    = 0.596   overall validation R²
R²(normalized)      = 0.596   pattern correlation (bias removed)

=== SEASONAL RANGE ===
r.sun   seasonal range: 6493 Wh/m²
SURFRAD seasonal range: 3433 Wh/m²

=== SPATIAL RANGE ===
r.sun   spatial range: 1364 Wh/m²
SURFRAD spatial range: 1958 Wh/m²


In [5]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf

# Add DOY-mean-centered and station-mean-centered columns
val["rsun_doy_mean"]    = val.groupby("doy")["rsun_Whm2"].transform("mean")
val["surfrad_doy_mean"] = val.groupby("doy")["surfrad_Whm2"].transform("mean")
val["rsun_sta_mean"]    = val.groupby("station")["rsun_Whm2"].transform("mean")
val["surfrad_sta_mean"] = val.groupby("station")["surfrad_Whm2"].transform("mean")

# Seasonal correlation: DOY means only (7 stations averaged)
doy_means = val.groupby("doy")[["rsun_Whm2","surfrad_Whm2"]].mean()
r2_seasonal = np.corrcoef(doy_means["rsun_Whm2"], doy_means["surfrad_Whm2"])[0,1]**2

# Spatial correlation: station means only (12 DOYs averaged)
sta_means = val.groupby("station")[["rsun_Whm2","surfrad_Whm2"]].mean()
r2_spatial = np.corrcoef(sta_means["rsun_Whm2"], sta_means["surfrad_Whm2"])[0,1]**2

# Within-station seasonal correlation (removes station effect)
r2_within_station = []
for st in val["station"].unique():
    sub = val[val["station"]==st]
    if len(sub) > 3:
        r2 = np.corrcoef(sub["rsun_Whm2"], sub["surfrad_Whm2"])[0,1]**2
        r2_within_station.append((st, round(r2,3), len(sub)))

# Within-DOY spatial correlation (removes DOY effect)
r2_within_doy = []
for doy in val["doy"].unique():
    sub = val[val["doy"]==doy]
    if len(sub) > 3:
        r2 = np.corrcoef(sub["rsun_Whm2"], sub["surfrad_Whm2"])[0,1]**2
        r2_within_doy.append((doy, round(r2,3), len(sub)))

print("=== SEPARATED SIGNAL DECOMPOSITION ===")
print(f"\nSeasonal R² (DOY means across 7 stations): {r2_seasonal:.3f}")
print(f"Spatial R²  (station means across 12 DOYs): {r2_spatial:.3f}")

print("\nWithin-station R² (seasonal signal per station):")
for st, r2, n in sorted(r2_within_station, key=lambda x: -x[1]):
    print(f"  {st:5s}: R²={r2:.3f}  (n={n})")

print("\nWithin-DOY R² (spatial signal per DOY):")
for doy, r2, n in sorted(r2_within_doy, key=lambda x: x[0]):
    print(f"  DOY{doy:03d}: R²={r2:.3f}  (n={n})")

=== SEPARATED SIGNAL DECOMPOSITION ===

Seasonal R² (DOY means across 7 stations): 0.955
Spatial R²  (station means across 12 DOYs): 0.435

Within-station R² (seasonal signal per station):
  dra  : R²=0.862  (n=12)
  sxf  : R²=0.833  (n=12)
  fpk  : R²=0.796  (n=12)
  tbl  : R²=0.781  (n=12)
  psu  : R²=0.588  (n=12)
  bon  : R²=0.515  (n=12)
  gwn  : R²=0.300  (n=12)

Within-DOY R² (spatial signal per DOY):
  DOY015: R²=0.712  (n=7)
  DOY045: R²=0.119  (n=7)
  DOY074: R²=0.503  (n=7)
  DOY105: R²=0.515  (n=7)
  DOY135: R²=0.074  (n=7)
  DOY166: R²=0.072  (n=7)
  DOY196: R²=0.009  (n=7)
  DOY227: R²=0.097  (n=7)
  DOY258: R²=0.158  (n=7)
  DOY288: R²=0.437  (n=7)
  DOY319: R²=0.728  (n=7)
  DOY349: R²=0.384  (n=7)
